# 15 · Transactions & ACID

A **transaction** groups several statements into one all-or-nothing unit.
- `BEGIN` starts it, `COMMIT` saves it, `ROLLBACK` undoes it.
- **ACID:** Atomicity, Consistency, Isolation, Durability.

The classic example is a money transfer: subtract from one account and add to
another. If the second step fails, the first must be undone — otherwise money
vanishes.

Because notebook SQL magic auto-commits each cell, this module uses a plain
Python `sqlite3` connection so we can control the transaction explicitly and
show `COMMIT` vs `ROLLBACK` clearly.

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

In [ ]:
import sqlite3, os

# Use a separate throwaway database so we never touch the course data.
path = 'data' if os.path.isdir('data') else '.'
demo_db = os.path.join(path, 'demo_bank.db')
conn = sqlite3.connect(demo_db)
conn.execute("DROP TABLE IF EXISTS accounts")
conn.execute("""CREATE TABLE accounts (
    name    TEXT PRIMARY KEY,
    balance REAL NOT NULL CHECK (balance >= 0)
)""")
conn.executemany("INSERT INTO accounts VALUES (?, ?)",
                 [('Alice', 100.0), ('Bob', 50.0)])
conn.commit()
print(conn.execute("SELECT * FROM accounts").fetchall())

## A successful transfer (COMMIT)
Move $30 from Alice to Bob atomically.

In [ ]:
try:
    conn.execute("BEGIN")
    conn.execute("UPDATE accounts SET balance = balance - 30 WHERE name = 'Alice'")
    conn.execute("UPDATE accounts SET balance = balance + 30 WHERE name = 'Bob'")
    conn.commit()
    print("Committed.")
except Exception as e:
    conn.rollback()
    print("Rolled back:", e)

print(conn.execute("SELECT * FROM accounts").fetchall())

## A failed transfer (ROLLBACK)
Now try to move $1000 from Bob (who only has $80). The `CHECK (balance >= 0)`
constraint rejects the debit, the exception triggers `ROLLBACK`, and **no
partial change** remains — that's Atomicity.

In [ ]:
try:
    conn.execute("BEGIN")
    conn.execute("UPDATE accounts SET balance = balance - 1000 WHERE name = 'Bob'")
    conn.execute("UPDATE accounts SET balance = balance + 1000 WHERE name = 'Alice'")
    conn.commit()
    print("Committed.")
except Exception as e:
    conn.rollback()
    print("Rolled back:", type(e).__name__, "-", e)

# Balances are unchanged from the previous cell:
print(conn.execute("SELECT * FROM accounts").fetchall())

## Manual ROLLBACK
You can also undo deliberately, e.g. after inspecting the result.

In [ ]:
conn.execute("BEGIN")
conn.execute("UPDATE accounts SET balance = 0 WHERE name = 'Alice'")
print("Inside txn:", conn.execute("SELECT * FROM accounts").fetchall())
conn.rollback()
print("After rollback:", conn.execute("SELECT * FROM accounts").fetchall())

## Clean up

In [ ]:
conn.close()
if os.path.exists(demo_db):
    os.remove(demo_db)
print("cleaned up")

### ✅ Recap
Transactions make multi-step changes atomic: `COMMIT` to save, `ROLLBACK` to
undo. ACID guarantees that your data stays correct even when things fail
midway.

**Next:** `16_capstone_project.ipynb` — put it all together.